In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "DeSTA-ntu/DeSTA-AQA5M-FROM-Llama3.1-8B-Instruct",
    cache_dir="/work/u1501463/hf_cache"
)



/home/u1501463/miniconda3/envs/gemini/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from pathlib import Path
import os

main_dataset_id = "VCTK-Corpus"
main_dataset_path = '/work/u1501463/VCTK'
main_dataset = dataset["train"].filter(
    lambda x: x["dataset"] == main_dataset_id
    # lambda x: x["dataset"] == "VCTK-Corpus"
)

def get_path(ds_path):

    result = ds_path.split("/")[-1]
    result = os.path.join(main_dataset_path,  result)

    return result

def get_dub_path(ds_path):
    filepath = ds_path.split("wav48/")[1]
    left_filepath = filepath.replace(".wav", '_mic1.flac')
    right_filepath = filepath.replace(".wav", '_mic2.flac')
    result_L = os.path.join(main_dataset_path, 'wav48_silence_trimmed', left_filepath)
    result_R = os.path.join(main_dataset_path, 'wav48_silence_trimmed', right_filepath)
    return result_L, result_R


In [51]:
import shutil
from merge_vocals import merge_vocals
import random
def sample_speech():
    idx = random.randint(0, len(main_dataset)-1)
    speech_data = main_dataset[idx]

    L, R = get_dub_path(speech_data['id'])
    shutil.copy(L, './sample_audios/L.flac')
    shutil.copy(R, 'sample_audios/R.flac')
    merge_vocals( './sample_audios/L.flac',  './sample_audios/R.flac', './sample_audios/speech.wav')

    speech_description = speech_data['seed_description']
    return speech_description

In [35]:
import json
def read_json(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    return data

noise_data = read_json('audioset_metadata.json')
noise_dataset_id = 'AudioSet-20K'
noise_dataset_path = '/work/u1501463/audioset_20k/20k'
noise_dataset = dataset["train"].filter(
    lambda x: x["dataset"] == noise_dataset_id
    # lambda x: x["dataset"] == "VCTK-Corpus"
)

def get_noise_path(noise_data_id, split):
    noise_data_id = 'Y' + noise_data_id
    print("id", noise_data_id, ',split', split)
    
    for item in noise_data:
        if item['id'] == noise_data_id:
            return os.path.join(noise_dataset_path, split, item['file name'])

In [54]:
def sample_noise():
    idx = random.randint(0, len(noise_dataset)-1)
    noise_metadata = noise_dataset[idx]
    noise_data_id = noise_metadata['id'].split("/")[-1].replace('.wav', '')
    data_split = noise_metadata['id'].split("/")[-3]
    noise_path = get_noise_path(noise_data_id, data_split)
    noise_description = noise_metadata['seed_description']
    shutil.copy(noise_path, './sample_audios/noise.wav')
    return noise_description

In [ ]:
from audio_mix import mix_audio
mix_audio('./sample_audios/speech.wav', './sample_audios/noise.wav', './merged.wav', snr_db = -5)

In [70]:
speech_description = sample_speech()
noise_description = sample_noise()
print(speech_description)
print(noise_description)
mix_audio('./sample_audios/speech.wav', './sample_audios/noise.wav', './sample_audios/merged.wav', snr_db = -5)

id YtS9TAK2TdkQ ,split train
[00:00-00:04] This time, there were no such rewards. (Gender: Female, Pitch:very low, Accent: us, Age: 26, Emotion: neutral, Speaker: p341)
[00:00-00:10] (crowd, battle cry)


'./sample_audios/merged.wav'

In [71]:
from gemini import gemini_inference
from train_QA_gen_prompt import QA_pair_gen_prompt

sample_path = '/home/u1501463/tool_use_LALM/sample_audios/merged.wav'
gen_QA_prompt = QA_pair_gen_prompt(
    n = 2,
    audio_token="<|audio_bos|><|AUDIO|><|audio_eos|>\n",
    description=speech_description, 
    background_description=noise_description, 
    # description=None, 
    # background_description=None,
    labels = None,
)

output = gemini_inference(gen_QA_prompt, "<|audio_bos|><|AUDIO|><|audio_eos|>\n", sample_path, 'low', False)
print(output.text)


[QUESTION]
What does the female speaker say over the sound of the crowd and battle cries?
A) This time, there were many such rewards.
B) Next time, we will get our rewards.
C) This time, there were no such rewards.
D) There were no such rewards left.
[/QUESTION]

[REASONING]
Step 1:
  Role: [Simplification]
  Why this tool is needed: To accurately transcribe the speech, we should isolate the speech from the background crowd and battle cries.
  Tool call:
```json
{
  "tool_call": {
    "name": "source_separation",
    "request_id": "sep_1",
    "parameters": {
      "audio_id": "0",
      "audio_begin": "00:00:00.000",
      "audio_end": "00:00:10.000",
      "stems": ["speech", "background noise"]
    }
  }
}
```
  Expected tool result:
<tool_result>
[TOOL_RESULT_BEGIN]
request_id: sep_1
tool: source_separation
status: success
stems:
  - label: speech
    audio: <audio_token_speech>
  - label: background noise
    audio: <audio_token_background>
[TOOL_RESULT_END]
</tool_result>
  Post-